# Selenium and scraping through heavy JavaScript

Let's revisit our diagram of how the web works updated for how modern websites built on react or
other fancy JavaScript frameworks work. Think about facebook or
[DeFlock](deflock.org) where you can interact with pages in complex ways. We
call sites like these _dynamic_ rather than a _static_ site where all of the content is stored in
the html.

![Modern Web](web2overview.png)

We've added the browser to our diagram as now the browser is doing a whole lot more than simply
formatting http requests and rendering html with css. Instead, on dynamic websites, servers send
back a whole load of JavaScript along with the html and css. The browser, then, must actually
do a bunch of computation. Often, the actual page content won't be stored in the html at all and
instead will only be loaded by once some JavaScript code retrieves it.

This is how we get websites that behave like applications and also makes scraping dynamic websites
a complete nightmare. If we were to use `requests` to try and scrape a _dynamic_ site, we would get
a bunch of `button`s and seemingly empty `div`s with content nowhere to be found.

In order to scrape pages that require a browser to actually render, we will simply automate a
browser.

## `selenium`

`selenium` is a tool for automating browsers. While not a scraping tool per se, `selenium` allows
us to fully render pages, click buttons, enter input, or do anything else we might need to do to
get the content of a dynamic page after running the JavaScript to load it.

Convienently, we can use `selenium` to run a browser in "headless" mode where we don't see a
broswer open up and do stuff but it still does all of the things a broswer can do. This gives us an
interface similar to any other library in Python even though we're actually running a browser and
mimicking mouse movements and inputs.

However, its both cool to look at and useful for development to just watch our automated browser do
its thing so for now we'll leave `selenium`s head intact.

We're going to learn how to use `selenium` by scraping a comradely website,
[Have I Been Flocked?](haveibeenflocked.com)

## Our goal for this workshop

Let's download the data from Have I Been Flocked? for all of the agencies in our states. For the
sake of example, we will do this for Michigan. We'll have to do a whole bunch of browser operations
to make this happen including manipulating dropdown menus, scrolling, and clicking buttons. This is
going to produce a _lot_ of data and despite that, we will not be a complete snapshot of Have I
Been Flocked? We'll do our best with a rough and ready approach with the goal of learning and
applying the concepts needed to work with selenium rather than building a perfect dataset.

## The basic cycle of `selenium`

 While some of this might seem straightforward to those of us versed in JavaScript, the way we
interact with elements on our target website is a little awkward. We'll see over and over again a
basic cycle consisting of selecting an element, loading the element, then doing something with it.

![The basic `selenium` process](selenium_loop.png)

The first step of openning a webpage is fairly obvious, when we get to the code we'll see just how
easy it is.

Its the selection step that can be the most awkward, and like our scraping with `requests`, it
requires a lot of using the developer tools in our browser of choice. There are a few ways of
selecting an element on a webpage in `selenium` but we'll use CSS classes.

The main use of CSS classes is to do things like "buttons in general should be grey but _this_
button should be red" by annotating the button in html with something like
`<button class="special-red-one>` paired with a block in the CSS saying
something like `button:special-red-one { bg-color: red }`. 

While we dont care about the use of these CSS classes for decoration, these classes can also
specify a particular, unique element on a webpage. Our browser's developer tools knows that we want
to do this and make it very easy to copy just the unique CSS specifiers for any element.

We'll get to the details of this soon but for now lets look at the other steps in the process.

Loading an element is often as simple as waiting just a little bit for all that fancy JavaScript to
do its thing. Sometimes, however, we may have to do something like scroll the _viewport_---all the
stuff inside the frame of the browser window---to a button so that its fully loaded and thus
clickable or to tell a dynamically loaded table to populate itself. This can be needed even if
we're running in headless mode and there is no viewer of the viewport as such.

Finally, we'll want to interact with the element in some way, maybe by clicking on something or by
pulling out its html content to process as we did when scraping with `requests`.

Without further ado, let's write some code.

## Our first steps with selenium

As we've done with other packages we must first add `selenium` to our `uv` configuration with
`uv add selenium`. Once thats done, we're ready to roll.

The primary object that we will be using do do stuff in our browser is a `webdriver` for some
browser of our choosing. We'll use Firefox here, but `selenium` also supports Chrome, Edge, and
maybe a couple of others. It shouldn't matter too much either way but frankly, Chrome is likely
your best bet as a lot of websites are built with it in mind. If you want a Chrome-like experience
without the Google, [selenium can use chromium in place of chrome with some minimal
effort](https://stackoverflow.com/a/18403389) (these instructions will make a lot more sense
shortly for what its worth).

In [2]:
from selenium import webdriver
import time  # we'll use this to add hard pauses
# from selenium.webdriver.chrome.options import Options
# opts = Options()
# opts.binary_location = "/usr/bin/chromium"  # `which chromium`
# driver = webdriver.Chrome(options=opts)


driver = webdriver.Firefox()

driver.get("https://haveibeenflocked.com/pd")

time.sleep(6)  # 5 second pause to see whats going on

driver.close()


When you run that code you should see a Firefox window open to Have I Been Flocked? and just sit
there for 5 seconds before closing.

Notice those little banners at the bottom of the page one for a newsletter signup and one with some
site status information. We don't need 'em. Lets click those little xs.

### Finding the selector for our button

In order to select things using _CSS selectors_ we'll need to use an object in selenium called
`By`. We'll also need to figure out what we need to match the CSS class of the X buttons. Here we
need to turn from our code and our fancy automated browsers to just a regular old browser. It
doesn't matter which one you are using, there should be some option for "Inspecting" whatever page
your on in the right click menu. The inspect function in your browser is extremely useful for many
things, for us we'll use it to identify which html elements contain the information or functions
we're looking for.

![How to find an elements CSS Selector](finding_css_selector.png)

Let's go through how we do this using the steps shown above:

1. Right click any where on the page and open the "Inspect" panels. On Firefox as shown, this opens
   on the bottom of the page while on Chromium-based browsers it opens to the side. You should not
   see a bunch of html in a tiny little panel somewhere.
2. Activate the select from page tool by clicking the button at the very top left of these new
   Inspect panels. Now when you move the cursor around you should see things get highlighted and
   description cards popping out to tell you what exactly is highlighted.
3. Find the `button` element that looks like it should cose the top banner. It may or may not
   matter whether we click the button itself as opposed to the X label on the button (which will be
   an `svg` element), but really we want to click the button. Once you find it, click on it and the
   select on page tool will no longer be active.
4. Move over to the html panel and notice that a button is now highlighted! Thats the html for the
   button we chose using the select on page tool. Right click it and copy the CSS Selector (or just
   Selector on chromium) for this button.

Ok, so now we have the mythical CSS Selector in our clipboard, we can paste it anywhere we can
paste text to see what this thing looks like. Once we're writing code, we'll see how to use it but
for now let's just take a look. Upon paste, you should see the text `button.absolute`. This means
nothing outside of its context but rest assured that this string is sufficnent for us to select
this particular button on our page.

If this felt weird and complicated, thats ok. We're going to do it a thousand more times.

### Click the damn button

In order to load an element we want to interact with we'll need to wait for which we'll need a pair
of tools, `WebDriverWait` and `expected_conditions`. `WebDriverWait` let's us set up an 
_explicit wait_ where our automated browser will wait for some condition for up to some
preset time. If itends up waiting longer than the preset time, it will throw an error and stop. As
might be clear from the name, `expected_conditons` contains a number of functions to help us set up
conditions for `WebDriverWait`.

Let's try to click this thing

In [3]:
from selenium import webdriver  # main selenium module for browser integration
from selenium.webdriver.common.by import By  # for selecting elements
from selenium.webdriver.support.ui import WebDriverWait  # for waiting
from selenium.webdriver.support import expected_conditions as EC  # for telling wait what to wait for
import time

driver = webdriver.Firefox()

# we wait using a wait object
wait = WebDriverWait(driver, timeout=10)  # 10 second max

driver.get("https://haveibeenflocked.com/pd")

# get the selector
button_selector = (By.CSS_SELECTOR, "button.absolute")

# select!
top_banner_closable = EC.element_to_be_clickable(button_selector)

# this gets us the ready to use button
close_button = wait.until(top_banner_closable)

# clicking is easy once we've done the hard part
close_button.click()

time.sleep(3)
driver.close()


So what happens if something goes wrong with us trying to click this button? Well the whole cell
stops running. This might not even be our fault! It could be that our browser just hiccups and
takes longer than our defined wait in order to load the button.

For this we'll use try and except.

In [4]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, timeout=10)

try:

    driver.get("https://haveibeenflocked.com/pd")

    top_banner_closable = EC.element_to_be_clickable((By.CSS_SELECTOR, "button.absolute"))
    close_button = wait.until(top_banner_closable)

    time.sleep(2)
    close_button.click()

except Exception as e:
    print(e)

time.sleep(2)
driver.close()

While we're doing housekeeping, let's put our button clicking logic into a function to make
everything a little easier to read as our code gets more complex

In [5]:
def do_click(css_target, waiter):

    # find the button and wait for it to load
    thing_to_click = EC.element_to_be_clickable((By.CSS_SELECTOR, css_target))
    clickable = waiter.until(thing_to_click)

    clickable.click()



Maybe we want to click something thats not exactly a button. Let's try clicking on the dropdown
menu for state with our new click function.


In [7]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, timeout=10)

try:

    driver.get("https://haveibeenflocked.com/pd")

    do_click("#state-filter", wait)

except Exception as e:
    print(e)

time.sleep(5)
driver.close()

So now we have an issue. We want to click on wisconsin of course, but in order to do that we'll
have to scroll. Maybe if we know the CSS class we need for wisconsin we can just stick it in our
`do_click()` function but often things will not be clickable unless they are in view.

Let's write a function we can use to scroll in our dropdown menu its going to be ugly.

In [8]:
from selenium.webdriver.common.actions.wheel_input import ScrollOrigin
from selenium.webdriver.common.action_chains import ActionChains

def do_scroll_element_until_click(scrollable_css,
                                  clickable_css,
                                  waiter,
                                  actions):

    # find the element we want to scroll in
    scrollable_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, scrollable_css)))

    # find the element we want to scroll to
    clickable_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, clickable_css)))

    # set where we want to scroll inside
    scroll_origin = ScrollOrigin.from_element(scrollable_element)

    while True:

        # scroll a little bit only on the y-axis (vertically)
        actions.scroll_from_origin(scroll_origin, 0, 50).perform()

        try:
            # click the thing if we can
            waiter.until(EC.element_to_be_clickable(clickable_element))
            clickable_element.click()

        except:
            # if we can't click yet, repeat the loop
            continue

        finally:
            # once we click, exit the loop
            break

Ok so we have the function, now let's use it. We'll click the dropdown menu then scroll down to
wisconsin and click that.

In [9]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, timeout=10)

# defining a new actions object
actions = ActionChains(driver)

try:
    
    driver.get("https://haveibeenflocked.com/pd")

    do_click("#state-filter", wait)

    time.sleep(2)

    do_scroll_element_until_click("#state-filter_list",
                                  "#state-filter_47",
                                  wait,
                                  actions)


except Exception as e:
    print(e)

time.sleep(5)
driver.close()

Let's take a little detour here and look at how we can download HTML using selenium like we did
with requests. Now that we have our page loaded, we can grab, for example, the list of all agencies
in this list and we dont even have to grab the whole page.

There is a div that contains all of the buttons, let's select it and pull out the html

In [10]:
from bs4 import BeautifulSoup

driver = webdriver.Firefox()
wait = WebDriverWait(driver, timeout=10)

# defining a new actions object
actions = ActionChains(driver)

try:
    
    driver.get("https://haveibeenflocked.com/pd")

    do_click("#state-filter", wait)

    time.sleep(2)

    do_scroll_element_until_click("#state-filter_list",
                                  "#state-filter_47",
                                  wait, actions)

    time.sleep(2)

    # same kind of find and wait operation as is in our functions
    agency_list = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.space-y-1")))
    agency_html = agency_list.get_attribute("outerHTML")

    # we can use beautifulsoup just like before!
    print(BeautifulSoup(agency_html).prettify())

except Exception as e:
    print(e)

time.sleep(5)
driver.close()

<div class="space-y-1 max-h-96 overflow-y-auto px-2 pt-2">
 <button class="bg-surface-elevated cursor-pointer hover:bg-surface-hover w-full text-left px-4 py-1 border border-border-default rounded-lg transition-colors">
  <div class="font-medium text-primary">
   [DEACTIVATED] - Richland Center WI PD
  </div>
 </button>
 <button class="bg-surface-elevated cursor-pointer hover:bg-surface-hover w-full text-left px-4 py-1 border border-border-default rounded-lg transition-colors">
  <div class="font-medium text-primary">
   Abbotsford WI PD
  </div>
 </button>
 <button class="bg-surface-elevated cursor-pointer hover:bg-surface-hover w-full text-left px-4 py-1 border border-border-default rounded-lg transition-colors">
  <div class="font-medium text-primary">
   Adams County WI SO
  </div>
 </button>
 <button class="bg-surface-elevated cursor-pointer hover:bg-surface-hover w-full text-left px-4 py-1 border border-border-default rounded-lg transition-colors">
  <div class="font-medium text-

From here we could parse our html as we have done before.

Theres one more thing we should look at, how to scroll the entire window not just inside some
element.

We'll write yet another function to scroll that is very similar to our previous function for
scrolling within an element.

In [12]:
def do_scroll_page_until_click(clickable_css, waiter, actions):

    element_to_click = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, clickable_css)))

    while True:

        actions.scroll_by_amount(0, 300).perform()

        try:

            # needed this on my machine on firefox
            actions.scroll_by_amount(0, 300).perform()
            
            waiter.until(EC.element_to_be_clickable(element_to_click))
            element_to_click.click()

        except:
            continue

        finally:
            break

Ok now we're going to jump over a lot of steps and see through downloading a csv full of audit logs
from this site.

In [13]:
driver = webdriver.Firefox()
wait = WebDriverWait(driver, timeout=10)

# defining a new actions object
actions = ActionChains(driver)

try:
    
    driver.get("https://haveibeenflocked.com/pd")

    do_click("button.shrink-0", wait)

    time.sleep(2)

    do_click("button.absolute", wait)

    time.sleep(2)

    do_click("#state-filter", wait)

    time.sleep(2)

    do_scroll_element_until_click("#state-filter_list",
                                  "#state-filter_47",
                                  wait, actions)

    time.sleep(2)

    do_scroll_element_until_click("div.space-y-1",
                                  "button.bg-surface-elevated:nth-child(230)",
                                  wait, actions)

    time.sleep(2)



    do_scroll_page_until_click("button.py-2",
                               wait,
                               actions)

    time.sleep(2)

    do_click("input.mt-0\\.5", wait)

    time.sleep(2)

    do_click("button.bg-accent", wait)

except Exception as e:
    print(e)

time.sleep(5)
driver.close()

Now its your turn. Try to use our functions to get the list of agencies that have performed
searches from every state starting from the same page as the previous examples. You will not need
all of the functions we have written and if it fails on a couple of states thats ok.

In [15]:

opts = webdriver.FirefoxOptions()

# uncomment this line to make the browser invisible
opts.add_argument("-headless")
# opts.add_argument("--headless=new")
driver = webdriver.Firefox(options = opts)



wait = WebDriverWait(driver, timeout=10)

for state_i in range(50):

    try:

        driver.get("https://haveibeenflocked.com/pd")

        # your stuff here
    
    except Exception as e:
        print(e)
        continue



driver.close()